In [1]:
%pip install pandas pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_parquet("reco_model_data.parquet")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:")
df.head(3)

Shape: (6477142, 11)

Columns: ['dt', 'session_id', 'event_name', 'product_id', 'image_url', 'productclass', 'class', 'subclass', 'designer', 'name', 'gender']

Dtypes:
 dt              object
session_id       int64
event_name      object
product_id       int64
image_url       object
productclass    object
class           object
subclass        object
designer        object
name            object
gender          object
dtype: object

First 3 rows:


,dt,session_id,event_name,product_id,image_url,productclass,class,subclass,designer,name,gender
0,2025-01-23,534295,click,2368,https://ounass.com//2/1/212588355_nocolor_in.j...,Beauty,"Creams, Moisturisers And Serums",Eye Treatments,Sensai,Cellular Performance Extra Intensive 10 Minute...,Unisex
1,2025-01-09,684505,click,2368,https://ounass.com//2/1/212588355_nocolor_in.j...,Beauty,"Creams, Moisturisers And Serums",Eye Treatments,Sensai,Cellular Performance Extra Intensive 10 Minute...,Unisex
2,2025-01-20,314905,click,2369,https://ounass.com//2/1/212588361_nocolor_in.j...,Beauty,Accessories And Tools,Face Brushes,Sensai,Cheek Brush,Womens


In [3]:
print("Nulls:\n", df.isnull().sum())
print("\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique — sample:", df[col].dropna().unique()[:5])

Nulls:
 dt                 0
session_id         0
event_name         0
product_id         0
image_url       3434
productclass     297
class              0
subclass           0
designer         800
name               0
gender          9781
dtype: int64

Unique values per column:
  dt: 31 unique — sample: ['2025-01-23' '2025-01-09' '2025-01-20' '2025-01-26' '2025-01-30']
  session_id: 169800 unique — sample: [534295 684505 314905 141190 459680]
  event_name: 1 unique — sample: ['click']
  product_id: 112145 unique — sample: [2368 2369 2370 2371 2372]
  image_url: 125406 unique — sample: ['https://ounass.com//2/1/212588355_nocolor_in.jpg?ts=1640726240.7545'
 'https://ounass.com//2/1/212588361_nocolor_in.jpg?ts=1640728805.1762'
 'https://ounass.com//2/1/212588549_nocolor_in.jpg?ts=1587289670.1673'
 'https://ounass.com//2/1/212588550_nocolor_in.jpg?ts=1587289672.6886'
 'https://ounass.com//2/1/212589468_navy_in.jpg?ts=1547034411.815']
  productclass: 7 unique — sample: ['Beauty' 'Clothing' 

In [4]:
df.sample(5)

,dt,session_id,event_name,product_id,image_url,productclass,class,subclass,designer,name,gender
1081845,2025-01-11,589890,click,28640,https://ounass.com//2/1/216429118_black_in.jpg...,Clothing,Knitwear,Dresses,Elisabetta Franchi,Midi Dress in Ribbed-knit,Womens
318666,2025-01-29,757855,click,8836,https://ounass.com//2/1/214586911_black_in.jpg...,Accessories,Small Leather Goods,Pouches,SAINT LAURENT,Uptown Pouch in Grain de Poudre Embossed Leath...,Womens
6461658,2025-01-25,797395,click,119920,https://ounass.com//2/1/218092164_red_in.jpg?t...,Clothing,Dresses,Long Sleeve,Meem Label,Rayna Midi Dress,Womens
2752356,2025-01-31,798655,click,71321,https://ounass.com//2/1/217429587_blk_in.jpg?t...,Clothing,Dresses,One Shoulder,Amri,Embellished Off-shoulder Maxi Dress in Stretch...,Womens
1701174,2025-01-20,819815,click,40421,https://ounass.com//2/1/216865119_red_in.jpg?t...,Clothing,Dresses,Sleeveless,Dolce & Gabbana,Sequinned Draped Maxi Dress,Womens


In [16]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet("reco_model_data.parquet")
print(f"Loaded: {df.shape[0]:,} rows, {df['product_id'].nunique():,} unique products, {df['session_id'].nunique():,} sessions")

Loaded: 6,477,142 rows, 112,145 unique products, 169,800 sessions


In [17]:
# Deduplicate more aggressively — same name + designer = same product, keep one
products = (
    df.sort_values('dt', ascending=False)
    .drop_duplicates(subset=['name', 'designer'])  # dedupe on name+designer not just product_id
    .drop_duplicates(subset='product_id')
    [['product_id', 'name', 'designer', 'productclass', 'class', 'subclass', 'gender', 'image_url']]
    .reset_index(drop=True)
)

for col in ['name', 'designer', 'productclass', 'class', 'subclass', 'gender']:
    products[col] = products[col].fillna('unknown').str.strip()

print(f"Product catalogue after dedup: {len(products):,} unique products")
print(f"\nProductclass distribution:")
print(products['productclass'].value_counts())

Product catalogue after dedup: 84,852 unique products

Productclass distribution:
productclass
Clothing       42079
Beauty         13312
Accessories    10320
Shoes           7683
Jewellery       5764
Bags            4759
Home             931
unknown            4
Name: count, dtype: int64


In [18]:
# 1. TF-IDF on product name + designer (captures brand + item specifics)
products['text_features'] = (
    products['designer'] + ' ' + 
    products['name'] + ' ' + 
    products['subclass']
)

tfidf = TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(products['text_features'])
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

# 2. One-hot encode categorical attributes
cat_features = ['productclass', 'class', 'gender']
ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore')
cat_matrix = ohe.fit_transform(products[cat_features])
print(f"Categorical matrix: {cat_matrix.shape}")

# 3. Combine — weight TF-IDF more heavily (0.7) vs categorical (0.3)
content_matrix = hstack([tfidf_matrix * 0.7, cat_matrix * 0.3])
print(f"Combined content matrix: {content_matrix.shape}")

TF-IDF matrix: (84852, 3000)
Categorical matrix: (84852, 153)
Combined content matrix: (84852, 3153)


In [19]:
# Products clicked together in the same session = collaborative signal
# Build session x product matrix
session_product = (
    df.groupby(['session_id', 'product_id'])
    .size()
    .reset_index(name='clicks')
)

# Pivot to session x product (binary — clicked or not)
# Use only sessions with 2+ product clicks (meaningful co-click signal)
session_counts = session_product.groupby('session_id')['product_id'].nunique()
valid_sessions = session_counts[session_counts >= 2].index
session_product_filtered = session_product[session_product['session_id'].isin(valid_sessions)]

print(f"Sessions with 2+ products: {len(valid_sessions):,}")
print(f"Co-click events: {len(session_product_filtered):,}")

# Map product_id to index in our catalogue
pid_to_idx = {pid: idx for idx, pid in enumerate(products['product_id'])}
products['idx'] = products['product_id'].map(pid_to_idx)

# Build sparse session-product matrix
from scipy.sparse import lil_matrix

all_pids = products['product_id'].tolist()
pid_set = set(all_pids)

# Filter to only products in our catalogue
spf = session_product_filtered[session_product_filtered['product_id'].isin(pid_set)].copy()
spf['pid_idx'] = spf['product_id'].map(pid_to_idx)

sessions = spf['session_id'].unique()
sess_to_idx = {s: i for i, s in enumerate(sessions)}
spf['sess_idx'] = spf['session_id'].map(sess_to_idx)

n_sessions = len(sessions)
n_products = len(products)

session_matrix = csr_matrix(
    (np.ones(len(spf)), (spf['sess_idx'], spf['pid_idx'])),
    shape=(n_sessions, n_products)
)
print(f"\nSession-product matrix: {session_matrix.shape}")
print(f"Density: {session_matrix.nnz / (n_sessions * n_products):.4%}")

Sessions with 2+ products: 145,687
Co-click events: 4,552,234

Session-product matrix: (143506, 84852)
Density: 0.0275%


In [20]:
# Build co-click counts properly using matrix multiplication
# product_session_matrix (n_products x n_sessions) @ its transpose = co-click counts

print("Building item-item co-click matrix...")

# Rebuild pid_to_idx after dedup
pid_to_idx = {pid: idx for idx, pid in enumerate(products['product_id'])}
products['idx'] = range(len(products))

pid_set = set(products['product_id'].tolist())
spf = session_product_filtered[session_product_filtered['product_id'].isin(pid_set)].copy()
spf['pid_idx'] = spf['product_id'].map(pid_to_idx)
spf = spf.dropna(subset=['pid_idx'])
spf['pid_idx'] = spf['pid_idx'].astype(int)

sessions = spf['session_id'].unique()
sess_to_idx = {s: i for i, s in enumerate(sessions)}
spf['sess_idx'] = spf['session_id'].map(sess_to_idx)

n_sessions = len(sessions)
n_products = len(products)

session_matrix = csr_matrix(
    (np.ones(len(spf)), (spf['sess_idx'], spf['pid_idx'])),
    shape=(n_sessions, n_products)
)

# Item-item co-click: normalise each product vector first
product_session_matrix = session_matrix.T  # n_products x n_sessions

# Normalise rows for cosine similarity
from sklearn.preprocessing import normalize
product_session_norm = normalize(product_session_matrix, norm='l2')

# We don't compute full NxN matrix (too big) — store normalised matrix
# and compute similarity on-the-fly per product in the recommend function
collab_matrix_norm = product_session_norm  # sparse, n_products x n_sessions

print(f"Co-click matrix: {collab_matrix_norm.shape}")
print(f"Products with at least 1 session click: {(collab_matrix_norm.sum(axis=1) > 0).sum():,}")
print(f"Products with NO session data (content-only fallback): {(collab_matrix_norm.sum(axis=1) == 0).sum():,}")

Building item-item co-click matrix...
Co-click matrix: (84852, 143506)
Products with at least 1 session click: 84,803
Products with NO session data (content-only fallback): 49


In [26]:
def get_recommendations(product_id, n=5, alpha=0.6, diversity_penalty=0.3, verbose=True):
    """
    diversity_penalty: how much to penalise products too similar to already-selected recs
    0.0 = no penalty (pure similarity ranking)
    0.3 = moderate diversity (default)
    0.6 = high diversity (very spread recommendations)
    """
    if product_id not in pid_to_idx:
        print(f"Product {product_id} not found")
        return None
    
    idx = pid_to_idx[product_id]
    product_info = products.iloc[idx]
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"INPUT: {product_info['designer']} — {product_info['name']}")
        print(f"       {product_info['productclass']} > {product_info['class']} > {product_info['subclass']} | {product_info['gender']}")
        print(f"{'='*70}")
    
    # Content similarity
    content_sim = cosine_similarity(content_matrix[idx], content_matrix).flatten()
    
    # Collaborative similarity
    collab_row = collab_matrix_norm[idx]
    has_collab = collab_row.nnz > 0
    
    if has_collab:
        collab_sim = (collab_matrix_norm @ collab_row.T).toarray().flatten()
        hybrid_sim = alpha * content_sim + (1 - alpha) * collab_sim
        mode = f"{int(alpha*100)}% content / {int((1-alpha)*100)}% collaborative"
    else:
        hybrid_sim = content_sim.copy()
        mode = "content-based only (cold start)"
    
    # Exclude self
    hybrid_sim[idx] = -1
    
    # ── Maximal Marginal Relevance (MMR) diversity selection ──
    # Instead of just top-N by score, iteratively pick the next product
    # that balances relevance vs similarity to already-selected items
    
    selected = []
    selected_indices = []
    scores = hybrid_sim.copy()
    
    # Build a candidate pool (top 50 by score)
    candidate_pool = np.argsort(scores)[::-1][:50]
    candidate_scores = scores[candidate_pool]
    
    for _ in range(n):
        if len(selected_indices) == 0:
            # First pick — just highest score
            best_pos = np.argmax(candidate_scores)
        else:
            # Compute similarity of each candidate to already-selected items
            selected_matrix = content_matrix[selected_indices]
            candidate_matrix = content_matrix[candidate_pool]
            sim_to_selected = cosine_similarity(candidate_matrix, selected_matrix).max(axis=1)
            
            # MMR score: relevance - penalty * max_similarity_to_selected
            mmr_scores = candidate_scores - diversity_penalty * sim_to_selected
            
            # Zero out already selected
            for si in selected_indices:
                if si in candidate_pool:
                    pos = np.where(candidate_pool == si)[0]
                    if len(pos) > 0:
                        mmr_scores[pos[0]] = -999
            
            best_pos = np.argmax(mmr_scores)
        
        best_idx = candidate_pool[best_pos]
        selected_indices.append(best_idx)
        selected.append({
            'rank': len(selected),
            'product_id': products.iloc[best_idx]['product_id'],
            'name': products.iloc[best_idx]['name'],
            'designer': products.iloc[best_idx]['designer'],
            'productclass': products.iloc[best_idx]['productclass'],
            'class': products.iloc[best_idx]['class'],
            'subclass': products.iloc[best_idx]['subclass'],
            'gender': products.iloc[best_idx]['gender'],
            'similarity_score': round(float(scores[best_idx]), 4)
        })
        # Remove from candidate pool
        candidate_pool = np.delete(candidate_pool, best_pos)
        candidate_scores = np.delete(candidate_scores, best_pos)
        
        if len(candidate_pool) == 0:
            break
    
    if verbose:
        print(f"\nTOP {n} RECOMMENDATIONS ({mode}, diversity_penalty={diversity_penalty}):\n")
        for rec in selected:
            print(f"  #{rec['rank']}  {rec['designer']} — {rec['name'][:65]}")
            print(f"       {rec['productclass']} > {rec['class']} > {rec['subclass']} | {rec['gender']}")
            print(f"       Score: {rec['similarity_score']:.4f}")
            print()
    
    return pd.DataFrame(selected)

print("MMR diversity-boosted recommendation function ready.")

MMR diversity-boosted recommendation function ready.


In [28]:
# Get a sample product from each major category to demonstrate
sample_products = {}
for pc in ['Clothing', 'Bags', 'Shoes', 'Beauty', 'Accessories']:
    sample = products[products['productclass'] == pc].sample(1)
    if len(sample) > 0:
        sample_products[pc] = sample.iloc[0]['product_id']

print("Sample products by category:")
for cat, pid in sample_products.items():
    p = products[products['product_id'] == pid].iloc[0]
    print(f"  {cat}: [{pid}] {p['designer']} — {p['name'][:50]}")

# Run recommendations for each
all_results = {}
for cat, pid in sample_products.items():
    print(f"\n{'#'*70}")
    print(f"CATEGORY: {cat}")
    results = get_recommendations(pid, n=5, alpha=0.6)
    all_results[cat] = results

Sample products by category:
  Clothing: [92303] Elliatt — Bethany Mini Dress in Crepe
  Bags: [99634] Strathberry — Mini Tote Bag in Leather
  Shoes: [7030] Private Collection — Peninsula Sandals in Ostrich Leather
  Beauty: [14496] Eucerin — pH5 Body Lotion, 400ml
  Accessories: [48774] Movado — Bold Titanium Watch, 45mm

######################################################################
CATEGORY: Clothing

INPUT: Elliatt — Bethany Mini Dress in Crepe
       Clothing > Dresses > Sleeveless | Womens

TOP 5 RECOMMENDATIONS (60% content / 40% collaborative, diversity_penalty=0.3):

  #0  Misha — Orlanda Sleeveless Mini Dress in Crepe
       Clothing > Dresses > Sleeveless | Womens
       Score: 0.5269

  #1  Elliatt — Inara Party Mini Dress
       Clothing > Dresses > Sleeveless | Womens
       Score: 0.5140

  #2  Theory — Shift Mini Dress in Japanese Crepe
       Clothing > Dresses > Sleeveless | Womens
       Score: 0.5030

  #3  Monot — Kristen Bow Mini Dress in Crepe
       Clo

In [29]:
# Evaluate coverage and intra-list diversity

def evaluate_model(n_samples=200, n_recs=5, alpha=0.6):
    """
    Coverage: % of catalogue that appears in recommendations
    Diversity: avg pairwise dissimilarity within recommendation lists
    Personalisation: how different recommendations are across input products
    """
    sample_pids = products['product_id'].sample(n_samples, random_state=42).tolist()
    
    all_rec_products = set()
    diversity_scores = []
    rec_sets = []
    
    for pid in sample_pids:
        recs = get_recommendations(pid, n=n_recs, alpha=alpha, verbose=False)
        if recs is not None and len(recs) > 0:
            rec_pids = recs['product_id'].tolist()
            all_rec_products.update(rec_pids)
            rec_sets.append(set(rec_pids))
            
            # Intra-list diversity: avg pairwise content dissimilarity
            indices = [pid_to_idx[p] for p in rec_pids if p in pid_to_idx]
            if len(indices) > 1:
                sim_matrix = cosine_similarity(content_matrix[indices])
                np.fill_diagonal(sim_matrix, 0)
                avg_sim = sim_matrix.sum() / (len(indices) * (len(indices)-1))
                diversity_scores.append(1 - avg_sim)  # diversity = 1 - similarity
    
    # Personalisation: how unique are rec lists across users
    personalisation_scores = []
    for i in range(min(50, len(rec_sets))):
        for j in range(i+1, min(50, len(rec_sets))):
            overlap = len(rec_sets[i] & rec_sets[j]) / n_recs
            personalisation_scores.append(1 - overlap)
    
    coverage = len(all_rec_products) / len(products)
    avg_diversity = np.mean(diversity_scores)
    avg_personalisation = np.mean(personalisation_scores)
    
    print(f"\n{'='*50}")
    print(f"MODEL EVALUATION (n={n_samples} samples, top-{n_recs})")
    print(f"{'='*50}")
    print(f"  Catalogue Coverage:    {coverage:.1%}  ({len(all_rec_products):,} unique products recommended)")
    print(f"  Intra-list Diversity:  {avg_diversity:.3f}  (higher = more varied recommendations)")
    print(f"  Personalisation:       {avg_personalisation:.3f}  (higher = more unique lists per user)")
    print(f"{'='*50}")
    
    return {
        'coverage': coverage,
        'diversity': avg_diversity,
        'personalisation': avg_personalisation
    }

metrics = evaluate_model()


MODEL EVALUATION (n=200 samples, top-5)
  Catalogue Coverage:    1.2%  (991 unique products recommended)
  Intra-list Diversity:  0.302  (higher = more varied recommendations)
  Personalisation:       1.000  (higher = more unique lists per user)


In [30]:
# Show effect of diversity penalty visually
print("DIVERSITY COMPARISON — Same input product, different penalty levels\n")

test_pid = products[products['productclass'] == 'Bags'].sample(1, random_state=7).iloc[0]['product_id']
p = products[products['product_id'] == test_pid].iloc[0]
print(f"Input: {p['designer']} — {p['name']}\n")

for penalty in [0.0, 0.3, 0.6]:
    print(f"\n── diversity_penalty={penalty} ──")
    recs = get_recommendations(test_pid, n=5, alpha=0.6, 
                                diversity_penalty=penalty, verbose=False)
    for _, r in recs.iterrows():
        print(f"  #{int(r['rank'])}  {r['designer']:<25} {r['name'][:45]:<45} [{r['subclass']}]")

DIVERSITY COMPARISON — Same input product, different penalty levels

Input: KARL LAGERFELD — Ikon Bucket Bag in Pebbled Faux Leather


── diversity_penalty=0.0 ──
  #0  KARL LAGERFELD            Ikon 2.0 Bucket Bag in Pebbled Faux Leather   [Small]
  #1  KARL LAGERFELD            Ikon Camera Bag in Pebbled Faux Leather       [Small]
  #2  KARL LAGERFELD            Ikon Shoulder Bag in Pebbled Faux Leather     [Small]
  #3  KARL LAGERFELD            K/Ikonik 2.0 Bucket Bag in Leather            [Small]
  #4  Tory Burch                Small Mcgraw Bucket Bag in Pebbled Leather    [Small]

── diversity_penalty=0.3 ──
  #0  KARL LAGERFELD            Ikon 2.0 Bucket Bag in Pebbled Faux Leather   [Small]
  #1  KARL LAGERFELD            K/Ikonik 2.0 Bucket Bag in Leather            [Small]
  #2  KARL LAGERFELD            Ikon Camera Bag in Pebbled Faux Leather       [Small]
  #3  Tory Burch                Small Mcgraw Bucket Bag in Pebbled Leather    [Small]
  #4  KARL LAGERFELD            Ik

In [31]:
# Demonstrate cold start handling
print("COLD START SCENARIO — New user, no session history")
print("Using alpha=1.0 (pure content-based)\n")

# Pick a product
test_pid = products[products['productclass'] == 'Bags'].sample(1).iloc[0]['product_id']
cold_start_recs = get_recommendations(test_pid, n=5, alpha=1.0)

print("\nRETURNING USER SCENARIO — Session history available")
print("Using alpha=0.5 (balanced hybrid)\n")
returning_recs = get_recommendations(test_pid, n=5, alpha=0.5)

print("\nCOMPARISON — Same product, different user context:")
print(f"{'Cold Start (content only)':<40} {'Returning User (hybrid)':<40}")
print("-" * 80)
for i in range(5):
    cs = cold_start_recs.iloc[i]['name'][:38] if cold_start_recs is not None else "N/A"
    ru = returning_recs.iloc[i]['name'][:38] if returning_recs is not None else "N/A"
    print(f"{cs:<40} {ru:<40}")

COLD START SCENARIO — New user, no session history
Using alpha=1.0 (pure content-based)


INPUT: Jil Sander — Flat Tote Bag in Cotton
       Bags > Top Handle > Medium | Mens

TOP 5 RECOMMENDATIONS (100% content / 0% collaborative, diversity_penalty=0.3):

  #0  Jil Sander — Medium Vertigo Tote Bag in Leather
       Bags > Top Handle > Medium | Womens
       Score: 0.6967

  #1  Jil Sander — Book Tote Bag in Canvas
       Bags > Carryall > Small | Mens
       Score: 0.6273

  #2  SAINT LAURENT — Rive Gauche Tote Bag in Linen & Leather
       Bags > Top Handle > Medium | Mens
       Score: 0.5635

  #3  Jil Sander — Woven Medium Tote in Leather
       Bags > Top Handle > Small | Womens
       Score: 0.6175

  #4  Jil Sander — Small Cannolo Bag in Raffia
       Bags > Top Handle > Medium | Womens
       Score: 0.5849


RETURNING USER SCENARIO — Session history available
Using alpha=0.5 (balanced hybrid)


INPUT: Jil Sander — Flat Tote Bag in Cotton
       Bags > Top Handle > Medium | Men

In [32]:
# Diagnostic — why is coverage low?

# 1. How many products have ANY session data?
products_with_clicks = df['product_id'].isin(products['product_id'])
click_counts = df[products_with_clicks].groupby('product_id').size().sort_values(ascending=False)

print("Click distribution across catalogue:")
print(f"  Products with 0 clicks:     {len(products) - len(click_counts):,}")
print(f"  Products with 1-5 clicks:   {((click_counts >= 1) & (click_counts <= 5)).sum():,}")
print(f"  Products with 6-20 clicks:  {((click_counts >= 6) & (click_counts <= 20)).sum():,}")
print(f"  Products with 21-100 clicks:{((click_counts >= 21) & (click_counts <= 100)).sum():,}")
print(f"  Products with 100+ clicks:  {(click_counts >= 100).sum():,}")
print(f"\nTop 10 most clicked products:")
top10 = click_counts.head(10).reset_index()
top10.columns = ['product_id', 'clicks']
top10 = top10.merge(products[['product_id','name','designer']], on='product_id')
print(top10[['designer','name','clicks']].to_string())

# 2. How concentrated are clicks?
total_clicks = click_counts.sum()
top100_clicks = click_counts.head(100).sum()
top1000_clicks = click_counts.head(1000).sum()
print(f"\nClick concentration:")
print(f"  Top 100 products = {top100_clicks/total_clicks:.1%} of all clicks")
print(f"  Top 1000 products = {top1000_clicks/total_clicks:.1%} of all clicks")
print(f"  Total products in catalogue: {len(products):,}")
print(f"  Total unique products clicked: {len(click_counts):,}")

Click distribution across catalogue:
  Products with 0 clicks:     0
  Products with 1-5 clicks:   17,549
  Products with 6-20 clicks:  26,929
  Products with 21-100 clicks:29,171
  Products with 100+ clicks:  11,347

Top 10 most clicked products:
          designer                                                   name  clicks
0       Mac Duggal                             Embroidered Gown in Jersey    3488
1  Needle & Thread           Bronte Off-Shoulder Ballerina Dress in Tulle    3418
2    Solace London                                 Kori Off-shoulder Gown    3084
3    Solace London       Rumi Halterneck Maxi Dress in Twill & Crepe-knit    2990
4    Solace London                         Kaila Cape Maxi Dress in Crepe    2964
5    Solace London  Alexis Off-shoulder Maxi Dress in Twill & Woven Crepe    2836
6   CHATS by C.Dam                                     Amara Draped Dress    2726
7   CHATS by C.Dam               Savi Maxi Dress in Stretch Double Jersey    2624
8             

### Question 8

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("ab_model_data.parquet")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:")
display(df.head(3))

print("\nNulls:\n", df.isnull().sum())

print("\nUnique values per column:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique — sample:", df[col].dropna().unique()[:5])

print("\nVariant distribution:")
print(df['variant'].value_counts())

print("\nDate range:", df['dt'].min(), "to", df['dt'].max())

print("\nEvent type columns (binary flags):")
for col in ['plp_view', 'plp_click', 'purchase']:
    print(f"  {col}: {df[col].value_counts().to_dict()}")

print("\nLogged-in distribution:")
print(df['is_logged_in'].value_counts())

print("\nUsers per variant:")
print(df.groupby('variant')['user_identifier'].nunique())

print("\nPurchase rate per variant (raw):")
print(df.groupby('variant')['purchase'].mean().round(4))

print("\nPLP click rate per variant (raw):")
print(df.groupby('variant')['plp_click'].mean().round(4))

print("\nPLP view rate per variant (raw):")
print(df.groupby('variant')['plp_view'].mean().round(4))

print("\nEvents per user per variant:")
print(df.groupby('variant')['user_identifier'].value_counts().groupby('variant').describe())

Shape: (14078419, 10)

Columns: ['dt', 'variant', 'country', 'ga_language', 'productclass', 'is_logged_in', 'user_identifier', 'plp_view', 'plp_click', 'purchase']

Dtypes:
 dt                 object
variant            object
country            object
ga_language        object
productclass       object
is_logged_in       object
user_identifier    object
plp_view            int64
plp_click           int64
purchase            int64
dtype: object

First 3 rows:


,dt,variant,country,ga_language,productclass,is_logged_in,user_identifier,plp_view,plp_click,purchase
0,2024-12-06,control,SA,Arabic,Bags,logged out,85f310bea532e4273dc682942f9ecdef41b12a844b8c6a...,1,0,0
1,2024-12-06,control,SA,Arabic,Clothing,logged out,9b729369e5c04f2dc1f27e2336e0f658b1bb7a55bb9e73...,1,0,0
2,2024-12-06,test,SA,Arabic,Accessories,logged out,eededf925f895050055e841494ace59ec2ad3f8ea8e7fd...,1,0,0



Nulls:
 dt                     0
variant                0
country                0
ga_language            0
productclass       21610
is_logged_in           0
user_identifier        0
plp_view               0
plp_click              0
purchase               0
dtype: int64

Unique values per column:
  dt: 10 unique — sample: ['2024-12-06' '2024-12-05' '2024-12-04' '2024-12-09' '2024-12-02']
  variant: 2 unique — sample: ['control' 'test']
  country: 7 unique — sample: ['SA' 'AE' 'QA' 'KW' 'Other']
  ga_language: 2 unique — sample: ['Arabic' 'English']
  productclass: 7 unique — sample: ['Bags' 'Clothing' 'Accessories' 'Jewellery' 'Home']
  is_logged_in: 2 unique — sample: ['logged out' 'logged in']
  user_identifier: 165694 unique — sample: ['85f310bea532e4273dc682942f9ecdef41b12a844b8c6acc33b89cc7ca1b0f47'
 '9b729369e5c04f2dc1f27e2336e0f658b1bb7a55bb9e73f0cd18b54ca0cca61b'
 'eededf925f895050055e841494ace59ec2ad3f8ea8e7fd6250820a65662031fb'
 'b122b3a56a66bd6f0795300c8a6f34ebee2162df28e3b